In [74]:
import pandas as pd
import numpy as np

In [75]:
file_path = "../data/Processed/furniture_returns_featured.xlsx"

df = pd.read_excel(file_path)

print("Feature engineered dataset loaded successfully!")
print("Dataset shape:", df.shape)

Feature engineered dataset loaded successfully!
Dataset shape: (5057, 23)


In [76]:
df.head()

,Return ID,Product,Value ₹,Transit Days,Return Reason,Condition Hint,Severity,Safety,SLA hrs,Inspection Outcome,...,Value Loss %,Severity Score,Safety Score,Condition Risk Score,Transit Risk Score,SLA Urgency Score,Value Risk Score,Priority Score,Priority,Priority Evidence
0,R-3047,bed,46133,2,size issue,no visible damage,unknown,low,24,no defect,...,2.852041,1,1,1,1,3,4,2.10,Medium,High product value; Urgent SLA
1,R2886,bed,40964,2,damaged,torn fabric,medium,high,48,repairable,...,29.927660,2,3,2,1,2,4,2.65,Medium,High product value; High safety risk
2,R2747,unknown,24559,1,damaged,chipped surface,medium,low,48,minor damage,...,12.385899,2,1,1,1,2,2,1.60,Low,Standard inspection priority
3,R4670,dining table,26326,1,defective,loose joints,low,low,48,minor damage,...,18.352503,1,1,2,1,2,2,1.45,Low,Standard inspection priority
4,R4898,dining table,26326,0,not as expected,color mismatch,low,low,48,minor damage,...,38.848679,1,1,1,1,2,2,1.40,Low,Standard inspection priority


In [77]:
required_columns = [

    "Return ID",

    "Value Risk Score",

    "Severity Score",

    "Safety Score",

    "Transit Risk Score",

    "SLA Urgency Score",

    "Condition Risk Score"

]

missing_columns = [

    column for column in required_columns

    if column not in df.columns

]

if missing_columns:

    print("Missing columns:", missing_columns)

else:

    print("All required columns are available!")

All required columns are available!


In [78]:
df["Baseline Score"] = (

    df["Value Risk Score"] * 0.30 +

    df["Severity Score"] * 0.20 +

    df["Safety Score"] * 0.20 +

    df["Transit Risk Score"] * 0.15 +

    df["SLA Urgency Score"] * 0.10 +

    df["Condition Risk Score"] * 0.05

)

df["Baseline Score"] = df["Baseline Score"].round(2)

print("Baseline scores calculated successfully!")

df[["Return ID", "Baseline Score"]].head(10)

Baseline scores calculated successfully!


,Return ID,Baseline Score
0,R-3047,2.10
1,R2886,2.65
2,R2747,1.60
3,R4670,1.45
4,R4898,1.40
5,R2596,2.75
6,R3729,2.60
7,R3755,1.85
8,R3354,1.85
9,r79,1.30


In [79]:
def assign_baseline_priority(score):
    
    if score >= 2.6:
        return "High"
    
    elif score >= 1.8:
        return "Medium"
    
    else:
        return "Low"

In [80]:
df["Baseline Priority"] = (
    df["Baseline Score"]
    .apply(assign_baseline_priority)
)

print(df["Baseline Priority"].value_counts())

Baseline Priority
Medium    2384
Low       2332
High       341
Name: count, dtype: int64


In [81]:
def create_baseline_evidence(row):
    
    reasons = []
    
    if row["Value Risk Score"] >= 4:
        reasons.append("High product value")
    
    if row["Severity Score"] >= 3:
        reasons.append("High damage severity")
    
    if row["Safety Score"] >= 3:
        reasons.append("High safety risk")
    
    if row["Transit Risk Score"] >= 3:
        reasons.append("Long transit duration")
    
    if row["SLA Urgency Score"] >= 3:
        reasons.append("Urgent SLA")
    
    if row["Condition Risk Score"] >= 3:
        reasons.append("Serious condition issue")
    
    if not reasons:
        reasons.append("Standard inspection requirement")
    
    return "; ".join(reasons)


df["Baseline Evidence"] = df.apply(
    create_baseline_evidence,
    axis=1
)

In [82]:
result_columns = [

    "Return ID",

    "Product",

    "Value ₹",

    "Severity",

    "Safety",

    "Transit Days",

    "SLA hrs",

    "Baseline Score",

    "Baseline Priority",

    "Baseline Evidence"

]

df[result_columns].head(20)

,Return ID,Product,Value ₹,Severity,Safety,Transit Days,SLA hrs,Baseline Score,Baseline Priority,Baseline Evidence
0,R-3047,bed,46133,unknown,low,2,24,2.10,Medium,High product value; Urgent SLA
1,R2886,bed,40964,medium,high,2,48,2.65,High,High product value; High safety risk
2,R2747,unknown,24559,medium,low,1,48,1.60,Low,Standard inspection requirement
3,R4670,dining table,26326,low,low,1,48,1.45,Low,Standard inspection requirement
4,R4898,dining table,26326,low,low,0,48,1.40,Low,Standard inspection requirement
5,R2596,bed,43374,high,high,0,72,2.75,High,High product value; High damage severity; High...
6,R3729,sofa,47719,high,medium,0,48,2.60,High,High product value; High damage severity
7,R3755,wardrobe,26326,low,medium,6,72,1.85,Medium,Long transit duration
8,R3354,coffee table,26326,medium,unknown,7,72,1.85,Medium,Long transit duration
9,r79,chair,26326,unknown,low,2,72,1.30,Low,Standard inspection requirement


In [83]:
priority_distribution = pd.DataFrame({
    "Count": df["Baseline Priority"].value_counts(),
    "Percentage": (
        df["Baseline Priority"]
        .value_counts(normalize=True) * 100
    ).round(2)
})

priority_distribution

,Count,Percentage
Baseline Priority,,
Medium,2384,47.14
Low,2332,46.11
High,341,6.74


In [84]:
high_priority_returns = df[
    df["Baseline Priority"] == "High"
]

high_priority_returns[
    [
        "Return ID",
        "Product",
        "Value ₹",
        "Severity",
        "Safety",
        "Transit Days",
        "SLA hrs",
        "Baseline Score",
        "Baseline Evidence"
    ]
].head(20)

,Return ID,Product,Value ₹,Severity,Safety,Transit Days,SLA hrs,Baseline Score,Baseline Evidence
1,R2886,bed,40964,medium,high,2,48,2.65,High product value; High safety risk
5,R2596,bed,43374,high,high,0,72,2.75,High product value; High damage severity; High...
6,R3729,sofa,47719,high,medium,0,48,2.60,High product value; High damage severity
12,R1245,wardrobe,30076,high,medium,9,72,2.60,High damage severity; Long transit duration; S...
18,R1261,bed,34776,high,high,4,24,2.80,High damage severity; High safety risk; Urgent...
66,R0136,sofa,54053,high,high,0,48,2.85,High product value; High damage severity; High...
78,R0146,bed,42976,low,medium,14,72,2.60,High product value; Long transit duration
83,R2753,sofa,52655,unknown,medium,6,24,2.65,High product value; Long transit duration; Urg...
116,R4372,bed,44361,high,medium,8,24,3.10,High product value; High damage severity; Long...
122,R1119,bed,41870,high,medium,0,48,2.70,High product value; High damage severity; Seri...


In [85]:
medium_priority_returns = df[
    df["Baseline Priority"] == "Medium"
]

medium_priority_returns[
    [
        "Return ID",
        "Value ₹",
        "Baseline Score",
        "Baseline Evidence"
    ]
].head(20)

,Return ID,Value ₹,Baseline Score,Baseline Evidence
0,R-3047,46133,2.10,High product value; Urgent SLA
7,R3755,26326,1.85,Long transit duration
8,R3354,26326,1.85,Long transit duration
10,R4559,44833,2.10,High product value; Urgent SLA
13,R4778,32474,1.90,Long transit duration
14,R3301,33083,1.85,Standard inspection requirement
16,R1821,30770,2.00,Long transit duration
20,R2444,31076,1.85,Standard inspection requirement
21,R-3402,3470,2.35,High damage severity; High safety risk; Long t...
23,R2380,50778,2.25,High product value; Urgent SLA


In [86]:
low_priority_returns = df[
    df["Baseline Priority"] == "Low"
]

low_priority_returns[
    [
        "Return ID",
        "Product",
        "Value ₹",
        "Baseline Score",
        "Baseline Evidence"
    ]
].head(20)

,Return ID,Product,Value ₹,Baseline Score,Baseline Evidence
2,R2747,unknown,24559,1.60,Standard inspection requirement
3,R4670,dining table,26326,1.45,Standard inspection requirement
4,R4898,dining table,26326,1.40,Standard inspection requirement
9,r79,chair,26326,1.30,Standard inspection requirement
11,R2281,wardrobe,26326,1.60,Serious condition issue
15,R1382,coffee table,11640,1.70,Long transit duration
17,R2452,chair,26326,1.50,Urgent SLA
19,R1885,unknown,6285,1.10,Standard inspection requirement
22,R4904,bookshelf,10795,1.75,High safety risk
24,R4390,chair,5624,1.10,Standard inspection requirement


In [87]:
df.groupby("Baseline Priority")[
    "Baseline Score"
].agg(
    ["count", "mean", "min", "max"]
).round(2)

,count,mean,min,max
Baseline Priority,,,,
High,341,2.77,2.6,3.35
Low,2332,1.43,1.0,1.75
Medium,2384,2.10,1.8,2.55


In [88]:
prioritized_returns = df.sort_values(
    by="Baseline Score",
    ascending=False
)

prioritized_returns[
    [
        "Return ID",
        "Product",
        "Baseline Score",
        "Baseline Priority",
        "Baseline Evidence"
    ]
].head(20)

,Return ID,Product,Baseline Score,Baseline Priority,Baseline Evidence
681,R2839,bed,3.35,High,High product value; High damage severity; High...
4398,R2638,bed,3.30,High,High product value; High damage severity; High...
383,R0460,bed,3.30,High,High product value; High damage severity; High...
3555,R1477,bed,3.30,High,High product value; High damage severity; High...
2611,R2519,bed,3.30,High,High product value; High damage severity; High...
3991,R0121,sofa,3.25,High,High product value; High damage severity; High...
2828,R1743,bed,3.25,High,High product value; High damage severity; High...
194,R1758,sofa,3.20,High,High product value; High safety risk; Long tra...
645,R0980,sofa,3.20,High,High product value; High damage severity; High...
4095,R0973,bed,3.20,High,High product value; High damage severity; High...


In [89]:
df = df.sort_values(
    by=["Baseline Score", "Value ₹"],
    ascending=[False, False]
).reset_index(drop=True)

df["Inspection Rank"] = df.index + 1

print(
    df[
        [
            "Inspection Rank",
            "Return ID",
            "Product",
            "Value ₹",
            "Baseline Score",
            "Baseline Priority"
        ]
    ].head(20)
)

    Inspection Rank Return ID       Product  Value ₹  Baseline Score  \
0                 1     R2839           bed    41143            3.35   
1                 2     R1477           bed    44311            3.30   
2                 3     R0460           bed    43938            3.30   
3                 4     R2519           bed    43798            3.30   
4                 5     R2638           bed    38884            3.30   
5                 6     R1743           bed    44482            3.25   
6                 7     R0121          sofa    37762            3.25   
7                 8     R0980          sofa    54891            3.20   
8                 9     R1758          sofa    49846            3.20   
9                10     R0973           bed    42542            3.20   
10               11     R0988           bed    47740            3.15   
11               12     R2937          sofa    45423            3.15   
12               13     R2744           bed    40089            

In [90]:
top_10 = df.sort_values(
    "Inspection Rank"
).head(10)

top_10[
    [
        "Inspection Rank",
        "Return ID",
        "Product",
        "Baseline Score",
        "Baseline Priority",
        "Baseline Evidence"
    ]
]

,Inspection Rank,Return ID,Product,Baseline Score,Baseline Priority,Baseline Evidence
0,1,R2839,bed,3.35,High,High product value; High damage severity; High...
1,2,R1477,bed,3.30,High,High product value; High damage severity; High...
2,3,R0460,bed,3.30,High,High product value; High damage severity; High...
3,4,R2519,bed,3.30,High,High product value; High damage severity; High...
4,5,R2638,bed,3.30,High,High product value; High damage severity; High...
5,6,R1743,bed,3.25,High,High product value; High damage severity; High...
6,7,R0121,sofa,3.25,High,High product value; High damage severity; High...
7,8,R0980,sofa,3.20,High,High product value; High damage severity; High...
8,9,R1758,sofa,3.20,High,High product value; High safety risk; Long tra...
9,10,R0973,bed,3.20,High,High product value; High damage severity; High...


In [91]:
output_path = "../data/Processed/furniture_returns_baseline.xlsx"

df.to_excel(
    output_path,
    index=False
)

print("Baseline prioritization results saved successfully!")
print(output_path)

Baseline prioritization results saved successfully!
../data/Processed/furniture_returns_baseline.xlsx


In [92]:
print("=" * 60)
print("BASELINE PRIORITIZER SUMMARY")
print("=" * 60)

print("\nTotal Returns:", len(df))

print("\nPriority Distribution:")
print(df["Baseline Priority"].value_counts())

print("\nTop 5 Returns for Inspection:")

print(
    df.sort_values("Inspection Rank")
    [
        [
            "Inspection Rank",
            "Return ID",
            "Product",
            "Baseline Priority",
            "Baseline Score"
        ]
    ]
    .head(5)
)

BASELINE PRIORITIZER SUMMARY

Total Returns: 5057

Priority Distribution:
Baseline Priority
Medium    2384
Low       2332
High       341
Name: count, dtype: int64

Top 5 Returns for Inspection:
   Inspection Rank Return ID Product Baseline Priority  Baseline Score
0                1     R2839     bed              High            3.35
1                2     R1477     bed              High            3.30
2                3     R0460     bed              High            3.30
3                4     R2519     bed              High            3.30
4                5     R2638     bed              High            3.30
